In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

FOLDER = "."

files = [f for f in os.listdir(FOLDER) if f.endswith(".csv")]
rows = []

for file in files:
    df = pd.read_csv(os.path.join(FOLDER, file), header=None)
    df.columns = ["pid", "arrival", "service", "start", "end"]
    scheduler = file.split("-")[0].upper()

    df["wait"]= df["start"] - df["arrival"]
    df["turnaround"]= df["end"]   - df["arrival"]
    total_time= df["end"].max() - df["start"].min()

    rows.append({
        "Scheduler":scheduler,
        "Avg Wait":df["wait"].mean(),
        "Avg Turnaround":df["turnaround"].mean(),
        "CPU Util %":df["service"].sum() / total_time * 100,
        "Throughput":len(df) / total_time,
        "P99 Wait":df["wait"].quantile(0.99),
    })

summary = pd.DataFrame(rows).sort_values("Avg Turnaround")
schedulers = summary["Scheduler"].tolist()
colors = ["#0080FF", "#FF5E00", "#00FF4C", "#FF0080", "#FFDD00", "#FF0000"][:len(schedulers)]

metrics = [
    ("Avg Wait","Average Wait Time"),
    ("Avg Turnaround", "Average Turnaround Time"),
    ("CPU Util %","CPU Utilisation %"),
    ("Throughput","Throughput (proc/unit)"),
    ("P99 Wait","P99 Wait Time"),
]

fig = make_subplots(rows=1, cols=len(metrics),
                    subplot_titles=[m[1] for m in metrics],
                    horizontal_spacing=0.09)

for col, (key, label) in enumerate(metrics, start=1):
    fig.add_trace(go.Bar(
        x=schedulers,
        y=summary[key],
        marker_color=colors,
        text=summary[key].round(2),
        showlegend=False,
    ), row=1, col=col)
    
fig.update_yaxes(range=[0, 500], row=1, col=1)
fig.update_yaxes(range=[0, 1500], row=1, col=2)
fig.update_yaxes(range=[0, 2300], row=1, col=5)

fig.update_layout(
    title="Scheduler Comparison",
    template="plotly_dark",
    height=500,
    width=1000,
)
fig.show()